In [3]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [4]:
!pip install -q datasets transformers trl peft accelerate bitsandbytes

In [5]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [6]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",  # LLaMA 3.1 8B
    max_seq_length = 2048,
    dtype = None,          # Auto-detect (bf16 sur A100, fp16 sur T4)
    load_in_4bit = True,   # QLoRA — indispensable sur T4
)

print("✅ Modèle chargé !")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


✅ Modèle chargé !


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Réduit la VRAM de 30%
    random_state = 42,
)

model.print_trainable_parameters()
# Affiche: ~1-2% des paramètres → très rapide à entraîner

Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [8]:
from datasets import load_dataset

# Charger depuis HuggingFace directement
dataset = load_dataset("gretelai/synthetic_text_to_sql", split="train")

print(f"✅ Dataset chargé : {len(dataset)} exemples")
print(dataset[0])  # Voir un exemple

README.md: 0.00B [00:00, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

✅ Dataset chargé : 100000 exemples
{'id': 5097, 'domain': 'forestry', 'domain_description': 'Comprehensive data on sustainable forest management, timber production, wildlife habitat, and carbon sequestration in forestry.', 'sql_complexity': 'single join', 'sql_complexity_description': 'only one join (specify inner, outer, cross)', 'sql_task_type': 'analytics and reporting', 'sql_task_type_description': 'generating reports, dashboards, and analytical insights', 'sql_prompt': 'What is the total volume of timber sold by each salesperson, sorted by salesperson?', 'sql_context': "CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 1

In [9]:
# Formater en prompt instruction → SQL
def format_prompt(example):
    prompt = f"""Vous êtes un expert SQL. Générez une requête SQL précise.

### Schéma de la base de données:
{example['sql_context']}

### Question:
{example['sql_prompt']}

### Requête SQL:
{example['sql']}"""
    return {"text": prompt}

dataset = dataset.map(format_prompt)

# Vérifier un exemple formaté
print(dataset[0]["text"])

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Vous êtes un expert SQL. Générez une requête SQL précise.

### Schéma de la base de données:
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');

### Question:
What is the total volume of timber sold by each salesperson, sorted by salesperson?

### Requête SQL:
SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;


In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from google.colab import drive

# Monter Google Drive pour sauvegardes
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/llama3-sql-output"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        # --- Batch & Gradient ---
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,

        # --- Scheduler ---
        warmup_steps = 50,
        max_steps = 500,          # ✅ Test rapide ~40min (commenter pour full)
        # num_train_epochs = 1,   # ← Décommenter pour entraînement complet

        # --- Optimiseur ---
        learning_rate = 2e-4,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",

        # --- Précision ---
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),

        # --- Logs & Save ---
        logging_steps = 25,
        save_steps = 250,          # ✅ Sauvegarde toutes les 250 steps
        save_total_limit = 2,      # ✅ Garde seulement les 2 derniers checkpoints
        output_dir = SAVE_DIR,     # ✅ Sauvegarde sur Google Drive

        seed = 42,
    ),
)

# Lancer l'entraînement
print("🚀 Début du fine-tuning...")
trainer.train()
print("✅ Fine-tuning terminé !")

Mounted at /content/drive
🚀 Début du fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
25,0.485043
50,0.437301
75,0.412279
100,0.460518
125,0.495987
150,0.488525
175,0.468386
200,0.477246
225,0.464323
250,0.476709


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llama3-sql-output/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llama3-sql-output/checkpoint-500/tokenizer_config.json.


✅ Fine-tuning terminé !


In [12]:
# Option A : Sauvegarder sur Google Drive
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/llama3-sql-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/llama3-sql-lora")
print("✅ Sauvegardé sur Google Drive !")

# Option B : Pousser sur HuggingFace Hub (public/privé)
# model.push_to_hub("ton-username/llama3-sql")
# tokenizer.push_to_hub("ton-username/llama3-sql")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llama3-sql-lora/tokenizer_config.json.


✅ Sauvegardé sur Google Drive !


In [14]:
from unsloth.chat_templates import get_chat_template

# Mode inférence rapide
FastLanguageModel.for_inference(model)

test_prompt = """Vous êtes un expert SQL. Générez une requête SQL précise.

### Schéma de la base de données:
CREATE TABLE orders (id INT, customer_id INT, amount FLOAT, date DATE);
CREATE TABLE customers (id INT, name TEXT, country TEXT);

### Question:
Quels sont les 5 clients français ayant dépensé le plus en 2024 ?

### Requête SQL:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 150,
    temperature = 0.1,
    do_sample = True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vous êtes un expert SQL. Générez une requête SQL précise.

### Schéma de la base de données:
CREATE TABLE orders (id INT, customer_id INT, amount FLOAT, date DATE);
CREATE TABLE customers (id INT, name TEXT, country TEXT);

### Question:
Quels sont les 5 clients français ayant dépensé le plus en 2024?

### Requête SQL:
SELECT c.name, SUM(o.amount) as total_spent FROM orders o JOIN customers c ON o.customer_id = c.id WHERE c.country = 'France' AND o.date BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY c.id ORDER BY total_spent DESC LIMIT 5;


In [15]:
from unsloth.chat_templates import get_chat_template

# Mode inférence rapide
FastLanguageModel.for_inference(model)

# Test avec un exemple
test_prompt = """Vous êtes un expert SQL. Générez une requête SQL précise.

### Schéma de la base de données:
CREATE TABLE employees (id INT, name TEXT, salary FLOAT, department TEXT);

### Question:
Quel est le salaire moyen par département ?

### Requête SQL:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 150,
    temperature = 0.1,
    do_sample = True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vous êtes un expert SQL. Générez une requête SQL précise.

### Schéma de la base de données:
CREATE TABLE employees (id INT, name TEXT, salary FLOAT, department TEXT);

### Question:
Quel est le salaire moyen par département?

### Requête SQL:
SELECT department, AVG(salary) FROM employees GROUP BY department;


In [19]:
# Télécharger le notebook directement
from google.colab import files

# Copier d'abord depuis Drive
import shutil
shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/ Fine-tune LLaMA 3 sur gretelai synthetic_text_to_sql.ipynb",
    "/content/finetune_llama3.ipynb"
)

# Télécharger
files.download("/content/finetune_llama3.ipynb")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# Cellule 1 — Installer et lancer le tunnel VS Code
!pip install -q colab-xterm
!wget -q "https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64" -O vscode_cli.tar.gz
!tar -xf vscode_cli.tar.gz

# Lancer le tunnel (te donnera un lien + code)
!./code tunnel --accept-server-license-terms

*
* Visual Studio Code Server
*
* By using the software, you agree to
* the Visual Studio Code Server License Terms (https://aka.ms/vscode-server-license) and
* the Microsoft Privacy Statement (https://privacy.microsoft.com/en-US/privacystatement).
*

  Visual Studio Code Tunnel v1.120.0

  ➜  Tunnel:   1788-e71b
  ➜  Open:  https://vscode.dev/tunnel/1788-e71b/content

[2026-05-13 17:09:10] info [tunnels::connections::relay_tunnel_host] Opened new client on channel 2
[2026-05-13 17:09:10] info [russh::server] wrote id
[2026-05-13 17:09:13] info [russh::server] read other id
[2026-05-13 17:09:13] info [russh::server] session is running
[2026-05-13 17:09:15] info [rpc.0] Checking /root/.vscode/cli/servers/Stable-0958016b2af9f09bb4257e0df4a95e2f90590f9f/log.txt and /root/.vscode/cli/servers/Stable-0958016b2af9f09bb4257e0df4a95e2f90590f9f/pid.txt for a running server...
[2026-05-13 17:09:15] info [rpc.0] Starting server...
[2026-05-13 17:09:15] info [rpc.0] Server started
[2026-05-13 17:09

In [17]:
# Nettoyer les métadonnées widgets corrompues
import json

notebook_path = "/content/drive/MyDrive/text_to_sql_llama3.ipynb"  # ← adapte le chemin

# Lire le notebook
with open(notebook_path, "r") as f:
    nb = json.load(f)

# Supprimer les widgets corrompus
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]
    print("✅ Widgets metadata supprimés")

# Sauvegarder le notebook nettoyé
with open(notebook_path, "w") as f:
    json.dump(nb, f, indent=1)

print("✅ Notebook nettoyé et sauvegardé !")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/text_to_sql_llama3.ipynb'